# PROJECT FORESIGHT: 03 ML Demand Forecasting & Inventory Risk Intelligence
**Client:** NorthBay Living


## 1. Setup & Imports


In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))


## 2. Load Evaluation Metrics & Model Comparison


In [ ]:
metrics_path = PROJECT_ROOT / 'outputs' / 'forecasts' / 'evaluation_metrics.csv'
metrics_df = pd.read_csv(metrics_path)

print('Model Comparison (Average across SKUs):')
print(metrics_df.groupby('model')[['mae', 'rmse', 'wape', 'mape']].mean())


## 3. Load 8-Week Forecasts & Confidence Intervals


In [ ]:
forecasts_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'forecasts' / 'forecast_8week.csv', parse_dates=['week_start'])
weekly_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'forecasts' / 'weekly_sales.csv', parse_dates=['week_start_date'])

sample_sku = 'SKU-001'
hist = weekly_df[weekly_df['sku_id'] == sample_sku].tail(16)
fc = forecasts_df[forecasts_df['sku_id'] == sample_sku]

plt.figure(figsize=(12, 5))
plt.plot(hist['week_start_date'], hist['weekly_units'], label='Historical Sales', marker='o', color='steelblue')
plt.plot(fc['week_start'], fc['forecast_units'], label='8-Week Forecast (GBR)', marker='s', color='crimson')
plt.fill_between(fc['week_start'], fc['lower_bound'], fc['upper_bound'], color='crimson', alpha=0.2, label='Confidence Interval')
plt.title(f'Demand Forecast with Confidence Intervals ({sample_sku})')
plt.xlabel('Week Start')
plt.ylabel('Units')
plt.legend()
plt.show()


## 4. Inventory Risk Assessment


In [ ]:
risk_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'risk' / 'stockout_risk.csv')
reorder_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'risk' / 'reorder_recommendations.csv')
overstock_df = pd.read_csv(PROJECT_ROOT / 'outputs' / 'risk' / 'overstock_analysis.csv')
markdown_candidates = pd.read_csv(PROJECT_ROOT / 'outputs' / 'risk' / 'markdown_candidates.csv')

with open(PROJECT_ROOT / 'outputs' / 'risk' / 'risk_summary.json', 'r') as f:
    risk_summary = json.load(f)

print('Stockout Risk Distribution:')
print(risk_df['risk_level'].value_counts())

print(f'\nTotal Revenue at Risk: ${risk_summary["revenue_at_risk"]["total_revenue_at_risk"]:,.2f}')
print(f'Total Excess Capital Locked: ${risk_summary["excess_capital"]["total_excess_capital"]:,.2f}')


## 5. Actionable Inventory Recommendations


In [ ]:
print('Top Reorder Recommendations:')
print(reorder_df.head(10))

print('\nMarkdown & Clearance Candidates:')
print(markdown_candidates.head(10))
